# Nisjemodell — nasjonal (raster + vektor)

Bygger et nasjonalt NiN/nisjemodell-datasett ved å kombinere nasjonale kilder,
etter samme metodikk som `notebooks/scratch/create_test_dataset_mr.ipynb`, men for
hele Norge og med disse endringene:

1. **Brakkvann** hentes fra det 50 m fylte rasteret `Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif`
   (samme padded 50 m-grid som substrat og fotisk sone).
2. **Substrat** hentes fra det modellerte rasteret
   `nisjedata-substrat-xgbclassifier_norge_latest_25833_2class_padded.tif` (reprojisert med
   WarpedVRT inn på gridet, som brakkvann).
3. **Tørrfall** håndteres i **vektor**-steget (ikke rasterisert på 50 m-gridet), fordi
   tørrfallspolygonene er små og detaljerte og vanskelige å gjengi presist på gridet.
4. **Observasjons-substrat** (`nisjedata-substrat-klassifisering`) legges på som
   **vektor**-overlay etter vektoriseringen, slik at de observerte polygongrensene
   bevares nøyaktig. Lys + salinitet arves fra det underliggende NiN-polygonet, og kun
   substrat-aksen byttes.

Pipeline:
Lys + Salinitet + Substrat → vindus-vis rasterklassifisering (3-bit nøkkel) →
vektorisering → tørrfall-overlay (vektor) →
observasjons-substrat-overlay (vektor) → trekk fra land → lagre.

In [1]:
import gc
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize as rio_rasterize
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.enums import Resampling

import mnk
import mnk.attributes
import mnk.sources
import mnk.utils
import mnk.vectorize

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Kilder og stier

- **Lys / referansegrid**: nasjonalt fotisk-sone-raster (50 m, EPSG:25833, 1=Eufotisk, 0=Afotisk, -1=nodata).
- **Brakkvann**: `hi/Norfjords_160_surf_salt_mean_UTM33_brakkvann.tif` (EPSG:32633, 1=brakk, 0=salt, 255=nodata).
- **Substrat**: nasjonal `nisjedata-substrat_norge` geoparquet (`BunnType`).
- **Tørrfall**: `mnk.sources.sea_map_torrfall()` (alle fylker) — brukes i vektor-steget.
- **Land**: nasjonal `Basisdata_Landareal` geoparquet.

In [2]:
REPO = Path.cwd()

photic_path   = "gs://niva-geodata/MarintNaturKart/results/nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833_padded.tif"
brakkvann_path = "gs://niva-geodata/MarintNaturKart/results/Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif"

# 0=soft, 1=mixed, 2=hard, 255=nodata
substrat_raster_path = "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-xgbclassifier_norge_latest_25833_2class_padded.tif"
# Authoritative polygons, overlaid as vectors at the end
obs_substrat_parquet = "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet"

# 1=DEM, 0=no DEM; per-polygon majority gives sannsynlighet_beskrivelse
dem_path = "gs://niva-geodata/MarintNaturKart/features/norge_is_dem.tif"

land_parquet = "gs://niva-geodata/MarintNaturKart/aux/land-mask-norway_FKB-osm-seaborder-subdivided-fullres_v3_25833_20260924.geo.parquet"

nin_raster_path = REPO / "nisjedata-modell_norge_2026_25833.tif"
nin_vec_path = REPO / "nisjedata-modell_norge_2026_25833_generalized.gpkg"

## NiN-koding

Vi koder tre binære dimensjoner (Lys, Salinitet, Substrat) til en 3-bit nøkkel og
antar foreløpig **Utenfor** tørrfall overalt. Tørrfall legges på i vektor-steget.

```
key = lys*4 + salinitet*2 + substrat
```

In [3]:
# NiN integer (1-12) -> (nin_kode_fkid, NiN_name); same as the MR notebook
NIN_INT_MAP = {
     1: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Euphotic Brackish Soft Inside
     2: ("NA-MA04", "Fjærebelte-sedimentbunn"),          # Euphotic Salt     Soft Inside
     3: ("NA-MC01", "Fast brakkvanns-fjærebeltebunn"),   # Euphotic Brackish Hard Inside
     4: ("NA-MA01", "Fast saltvanns-fjærebeltebunn"),    # Euphotic Salt     Hard Inside
     5: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Euphotic Brackish Soft Outside
     6: ("NA-MC04", "Brakkvanns-sedimentbunn"),          # Aphotic  Brackish Soft Outside
     7: ("NA-MA05", "Eufotisk saltvanns-sedimentbunn"),  # Euphotic Salt     Soft Outside
     8: ("NA-MA06", "Afotisk saltvanns-sedimentbunn"),   # Aphotic  Salt     Soft Outside
     9: ("NA-MC02", "Fast brakkvannsbunn"),              # Euphotic Brackish Hard Outside
    10: ("NA-MC02", "Fast brakkvannsbunn"),              # Aphotic  Brackish Hard Outside
    11: ("NA-MA02", "Eufotisk fast saltvannsbunn"),      # Euphotic Salt     Hard Outside
    12: ("NA-MA03", "Afotisk fast saltvannsbunn"),       # Aphotic  Salt     Hard Outside
}

# 3-bit LUT (tørrfall = outside):  key = lys*4 + salinitet*2 + substrat
#   0: Aphotic  Salt     Soft -> 8     1: Aphotic  Salt     Hard -> 12
#   2: Aphotic  Brackish Soft -> 6     3: Aphotic  Brackish Hard -> 10
#   4: Euphotic Salt     Soft -> 7     5: Euphotic Salt     Hard -> 11
#   6: Euphotic Brackish Soft -> 5     7: Euphotic Brackish Hard -> 9
LUT3 = np.array([8, 12, 6, 10, 7, 11, 5, 9], dtype=np.uint8)

# Only euphotic "outside" codes change; aphotic codes (6,8,10,12) are untouched
# since tørrfall is intertidal/euphotic.
TORRFALL_REMAP = {5: 1, 7: 2, 9: 3, 11: 4}

# Observation overlay swaps ONLY the substrate axis of an existing NiN code.
# Soft <-> hard pairs with the same light/salinity/zone:
#   (1,3) (2,4) euphotic inside;  (5,9) (7,11) euphotic outside;  (6,10) (8,12) aphotic.
SUBSTRAT_TO_FAST = {1: 3, 2: 4, 5: 9, 6: 10, 7: 11, 8: 12}
SUBSTRAT_TO_LOST = {v: k for k, v in SUBSTRAT_TO_FAST.items()}
print(LUT3)

[ 8 12  6 10  7 11  5  9]


## Last observasjons-substrat (nasjonalt)

Substrat for selve gridet hentes fra det modellerte rasteret i loopen under.
Her lastes kun **observasjons**-polygonene (`nisjedata-substrat-klassifisering`),
som legges på som vektor-overlay etter vektoriseringen for å bevare geometrien.


In [4]:
BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=soft, 1=hard

gdf_obs = gpd.read_parquet(obs_substrat_parquet).to_crs(CRS)
gdf_obs = gdf_obs[gdf_obs["BunnType"].isin(BUNN_TO_INT)].reset_index(drop=True)
gdf_obs["_obs_fast"] = gdf_obs["BunnType"].map(BUNN_TO_INT).astype("uint8")
gdf_obs = gdf_obs[["geometry", "_obs_fast"]]
_ = gdf_obs.sindex  # build spatial index
print(f"Observasjons-polygoner: {len(gdf_obs):,}")

Observasjons-polygoner: 130,919


## Vindus-vis rasterklassifisering

For hvert rad-stripe-vindu:
1. les Lys fra referansegridet,
2. reprojiser brakkvann (WarpedVRT) inn på gridet,
3. reprojiser modellert substrat-raster (WarpedVRT) inn på gridet
   (2=fastbunn → Fast, ellers Løst),
4. kod til NiN (Utenfor-antakelse) og skriv blokk til midlertidig GeoTIFF.

In [ ]:
BLOCK_ROWS = 2048

with rasterio.open(photic_path) as photic:
    transform = photic.transform
    width, height = photic.width, photic.height
    photic_nodata = photic.nodata if photic.nodata is not None else -1

    profile = dict(
        driver="GTiff", height=height, width=width, count=1, dtype="uint8",
        crs=CRS, transform=transform, nodata=NODATA, compress="deflate",
        tiled=True, blockxsize=512, blockysize=512,
    )

    brak_src = rasterio.open(brakkvann_path)
    brak_vrt = WarpedVRT(
        brak_src, crs=CRS, transform=transform, width=width, height=height,
        resampling=Resampling.nearest, src_nodata=brak_src.nodata, nodata=255,
    )

    sub_src = rasterio.open(substrat_raster_path)
    sub_vrt = WarpedVRT(
        sub_src, crs=CRS, transform=transform, width=width, height=height,
        resampling=Resampling.nearest, src_nodata=sub_src.nodata, nodata=255,
    )

    counts = np.zeros(13, dtype=np.int64)
    with rasterio.open(nin_raster_path, "w", **profile) as dst:
        for row0 in range(0, height, BLOCK_ROWS):
            nrows = min(BLOCK_ROWS, height - row0)
            win = Window(0, row0, width, nrows)
            shp = (nrows, width)

            lys_raw = photic.read(1, window=win)
            lys_valid = lys_raw != photic_nodata
            lys = (lys_raw == 1).astype(np.uint8)  # 1=euphotic, 0=aphotic

            # Brackish only where VRT==1; coverage gaps count as salt
            brak = brak_vrt.read(1, window=win)
            salinitet = (brak == 1).astype(np.uint8)

            # 2=hard -> 1, 0/1 -> soft (0)
            sub_raw = sub_vrt.read(1, window=win)
            sub_valid = sub_raw != 255
            substrat = (sub_raw == 2).astype(np.uint8)

            valid = lys_valid & sub_valid
            nin = np.full(shp, NODATA, dtype=np.uint8)
            key = (lys[valid] * 4 + salinitet[valid] * 2 + substrat[valid]).astype(np.uint8)
            nin[valid] = LUT3[key]
            dst.write(nin, 1, window=win)

            for v in np.unique(nin[nin != NODATA]):
                counts[v] += int(np.sum(nin == v))
            print(f"  rows {row0:>6}-{row0+nrows:<6} valid={int(valid.sum()):>12,}", end="\r")

    brak_vrt.close()
    brak_src.close()
    sub_vrt.close()
    sub_src.close()

print("\nNiN-fordeling (før tørrfall/observasjon/land):")
for i, (code_, name) in NIN_INT_MAP.items():
    if counts[i]:
        print(f"  {i:2d} {code_} {name}: {counts[i]:,}")

## Vektoriser + generaliser (GRASS)

Vektoriserer NiN-rasteret og fjerner 50 m trappetrinn-artefaktene i samme, disk-streamede operasjon via GRASS `r.to.vect` → `v.generalize` (Douglas–Peucker, terskel 25 m). Topologi bevares slik at felles grenser mellom NiN-klasser ikke får glipper/overlapp. Minnegjerrig fordi GRASS strømmer mot disk.

In [ ]:
mnk.vectorize.with_grass(nin_raster_path, nin_vec_path)

In [5]:
gdf_nin = gpd.read_file(nin_vec_path)
gdf_nin = gdf_nin.rename(columns={"class_int": "nin_int"})
gdf_nin = gdf_nin[gdf_nin["nin_int"] != NODATA].reset_index(drop=True)
gdf_nin = gdf_nin.explode(index_parts=False).reset_index(drop=True)
if gdf_nin.crs is None or gdf_nin.crs.to_epsg() != EPSG:
    gdf_nin = gdf_nin.set_crs(CRS)

# Merge thin generalization slivers into their largest same-class neighbour
# BEFORE the vector overlays, so slivers are not propagated and the overlays
# run faster.
gdf_nin = mnk.vectorize.merge_slivers(
    gdf_nin, min_area=2500, max_thinness=0.1, class_col="nin_int"
)
# Majority DEM coverage per polygon, carried as a temp flag to sannsynlighet_beskrivelse
gdf_nin["_is_dem"] = mnk.utils.dem_majority(gdf_nin, dem_path)

gdf_nin[["nin_kode_fkid", "NiN_name"]] = gdf_nin["nin_int"].map(NIN_INT_MAP).apply(pd.Series)
print(f"Polygoner: {len(gdf_nin):,}")
gdf_nin.head()

Polygoner: 350,270


,cat,nin_int,label,geometry,_is_dem,nin_kode_fkid,NiN_name
0,1,11,,"POLYGON ((972200 7942350, 972225 7942300, 9722...",False,NA-MA02,Eufotisk fast saltvannsbunn
1,2,7,,"POLYGON ((970800 7941400, 970825 7941350, 9709...",True,NA-MA05,Eufotisk saltvanns-sedimentbunn
2,3,11,,"POLYGON ((972700 7942400, 972700 7942325, 9727...",False,NA-MA02,Eufotisk fast saltvannsbunn
3,4,11,,"POLYGON ((967100 7939600, 967100 7939525, 9670...",False,NA-MA02,Eufotisk fast saltvannsbunn
4,5,7,,"POLYGON ((973550 7942400, 973575 7942300, 9736...",False,NA-MA05,Eufotisk saltvanns-sedimentbunn


## Tørrfall i vektor-steget

Tørrfallspolygonene er små og detaljerte. I stedet for å rasterisere dem til 50 m
legger vi dem på som vektor-overlay: NiN-polygoner (eufotiske) som overlapper tørrfall
splittes, og de delene som ligger **innenfor** tørrfall remappes:

`NA-MA05→NA-MA04`, `NA-MC02→NA-MC01`, `NA-MA02→NA-MA01`, `NA-MC04→NA-MC04`.

In [6]:
gdf_torrfall = mnk.sources.sea_map_torrfall()  # all counties
gdf_torrfall = gdf_torrfall.to_crs(CRS)[["geometry"]].reset_index(drop=True)
print(f"Tørrfall-polygoner: {len(gdf_torrfall):,}")

Tørrfall-polygoner: 214,779


In [7]:
# Only codes that tørrfall actually changes need splitting
remap_mask = gdf_nin["nin_int"].isin(TORRFALL_REMAP)
gdf_remap = gdf_nin[remap_mask].reset_index(drop=True)
gdf_keep = gdf_nin[~remap_mask].reset_index(drop=True)

# Inside tørrfall: intersect and remap the NiN code
inside = mnk.vectorize.intersect_layers(gdf_remap, gdf_torrfall)
inside["nin_int"] = inside["nin_int"].map(TORRFALL_REMAP).astype(gdf_remap["nin_int"].dtype)

# Outside tørrfall: difference, code unchanged
outside = mnk.vectorize.subtract_layer(gdf_remap, gdf_torrfall)

gdf_nin_tf = pd.concat([gdf_keep, outside, inside], ignore_index=True)
gdf_nin_tf = gpd.GeoDataFrame(gdf_nin_tf, geometry="geometry", crs=CRS)
gdf_nin_tf = gdf_nin_tf[~gdf_nin_tf.is_empty].explode(index_parts=False).reset_index(drop=True)
gdf_nin_tf[["nin_kode_fkid", "NiN_name"]] = gdf_nin_tf["nin_int"].map(NIN_INT_MAP).apply(pd.Series)
print(f"Polygoner etter tørrfall-overlay: {len(gdf_nin_tf):,}")

# Free intermediates to keep peak memory down (re-run from cell 12 to restore)
del gdf_nin, gdf_torrfall, gdf_remap, gdf_keep, inside, outside, remap_mask
gc.collect()

Polygoner etter tørrfall-overlay: 867,895


0

## Observasjons-substrat i vektor-steget

Observasjonspolygonene er autoritative for substrat. De legges på som vektor-overlay
slik at de observerte grensene bevares nøyaktig (ingen piksel-snapping):

1. `intersection` med NiN-laget → hver bit arver lys+salinitet fra det underliggende
   polygonet (store observasjoner splittes kun langs lys/salinitet-grenser),
2. kun substrat-aksen byttes via `SUBSTRAT_TO_FAST` / `SUBSTRAT_TO_LOST`,
3. `difference` stanser observasjonsfotavtrykket ut av NiN-laget, og de byttede
   observasjonsbitene limes inn igjen.
4. tilgrensende observasjonsbiter med samme NiN-kode slås sammen (dissolve), slik at
   observasjonene ikke blir unødvendig oppsplittet.


In [8]:
_dtype = gdf_nin_tf["nin_int"].dtype

# Inherit light+salinity from the underlying NiN polygon; only substrate is swapped
obs_pieces = mnk.vectorize.intersect_layers(gdf_obs, gdf_nin_tf)

In [9]:
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype("int64")
_fast = obs_pieces["_obs_fast"] == 1
obs_pieces.loc[_fast, "nin_int"] = (
    obs_pieces.loc[_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_FAST.get(v, v))
)
obs_pieces.loc[~_fast, "nin_int"] = (
    obs_pieces.loc[~_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_LOST.get(v, v))
)
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype(_dtype)

# Dissolve after the overlay so the stamp-out below still uses the small
# original observation polygons (much faster)
n_before = len(obs_pieces)
obs_pieces = (
    obs_pieces[["nin_int", "geometry"]]
    .dissolve(by="nin_int", as_index=False)
    .explode(index_parts=False)
    .reset_index(drop=True)
)
print(f"Observasjonsbiter: {n_before:,} -> {len(obs_pieces):,} etter sammenslåing")

# Stamp the observation footprint out of the NiN layer, then add the swapped pieces
nin_minus = mnk.vectorize.subtract_layer(gdf_nin_tf, gdf_obs[["geometry"]])

nin_minus["_substrat_kilde"] = "substrat_grunn_modellert"
obs_pieces["_substrat_kilde"] = "substrat_kartlagt_ngu"

gdf_nin_obs = pd.concat([nin_minus, obs_pieces], ignore_index=True)
gdf_nin_obs = gpd.GeoDataFrame(gdf_nin_obs, geometry="geometry", crs=CRS)
gdf_nin_obs = gdf_nin_obs[~gdf_nin_obs.is_empty].explode(index_parts=False).reset_index(drop=True)
gdf_nin_obs[["nin_kode_fkid", "NiN_name"]] = gdf_nin_obs["nin_int"].map(NIN_INT_MAP).apply(pd.Series)
gdf_nin_obs = gdf_nin_obs[["nin_int", "nin_kode_fkid", "NiN_name", "_substrat_kilde", "_is_dem", "geometry"]]
print(f"Polygoner etter observasjons-overlay: {len(gdf_nin_obs):,}")

del gdf_nin_tf, gdf_obs, obs_pieces, nin_minus
gc.collect()

Observasjonsbiter: 343,172 -> 221,850 etter sammenslåing
Polygoner etter observasjons-overlay: 1,123,687


0

## Trekk fra land

## Klipp til grunnlinje (1 nautisk mil)

Fjern polygoner utenfor grunnlinje + 1 nautisk mil, før land trekkes fra.


In [10]:
print(f"Polygoner før grunnlinje-klipp: {len(gdf_nin_obs):,}")
gdf_nin_obs = mnk.vectorize.clip_to_grunnlinje(gdf_nin_obs)
print(f"Polygoner etter grunnlinje-klipp: {len(gdf_nin_obs):,}")

Polygoner før grunnlinje-klipp: 1,123,687


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/osgeo/gdal.py:13117: RuntimeWarning: A geometry of type MULTIPOLYGON is inserted into layer data of geometry type POLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  return _gdal.wrapper_GDALVectorTranslateDestName(*args)


Polygoner etter grunnlinje-klipp: 1,123,207


In [11]:
land = gpd.read_parquet(land_parquet)
land = land.to_crs(CRS)

gdf_nin_land = mnk.vectorize.subtract_land(gdf_nin_obs, land)
print(f"Polygoner etter land-subtraksjon: {len(gdf_nin_land):,}")

del land, gdf_nin_obs
gc.collect()

Subtracting land from 500,920 of 1,123,207 polygons...
Polygoner etter land-subtraksjon: 1,099,066


0

## Lagre resultat

In [12]:
gdf_nin_land.to_file(str(nin_vec_path).replace(".gpkg", "_clipped.gpkg"))

## Legg til Naturkart-attributter

Avled datamodell-felt (nin_flate_NK) som kan utledes nå. `datakilde` er pipe-separert
(`|`) og kan inneholde flere kilder: `lys_modellert_copernicus_S3` (lys, alltid)
og substratkilden — `substrat_grunn_modellert` (modellert) eller
`substrat_kartlagt_ngu` (observasjoner), f.eks.
`lys_modellert_copernicus_S3|substrat_kartlagt_ngu`.

`sannsynlighet_beskrivelse`: `høy` for observert substrat (NGU), `medium` der
vi har DEM-data (majoritet av pikslene i `features/norge_is_dem.tif`),
ellers `lav`.


In [5]:
gdf_nin_land = mnk.attributes.add_nkm_attributes(gdf_nin_land)
gdf_nin_land = gdf_nin_land.drop(columns=["_substrat_kilde", "_is_dem"])
gdf_nin_land = gdf_nin_land[[
    "nk_fid", "opphav_kode", "nin_int", "nin_kode_fkid", "NiN_name",
    "nkm_prosess", "nkm_lys", "nkm_salinitet", "nkm_substrat", "nkm_toerfall",
    "metode_klassifisering", "datakilde", "sannsynlighet_beskrivelse", "geometry",
]]
print(gdf_nin_land["datakilde"].value_counts())
print(gdf_nin_land["sannsynlighet_beskrivelse"].value_counts())
gdf_nin_land.head()

datakilde
lys_modellert_copernicus_S3|substrat_grunn_modellert    881863
lys_modellert_copernicus_S3|substrat_kartlagt_ngu       217203
Name: count, dtype: int64
sannsynlighet_beskrivelse
lav       519581
medium    362282
høy       217203
Name: count, dtype: int64


,nk_fid,opphav_kode,nin_int,nin_kode_fkid,NiN_name,nkm_prosess,nkm_lys,nkm_salinitet,nkm_substrat,nkm_toerfall,metode_klassifisering,datakilde,sannsynlighet_beskrivelse,geometry
0,NKM-0000000001,NKM,8,NA-MA06,Afotisk saltvanns-sedimentbunn,A,afotisk,saltvann,loesbunn,permanent_vanndekket,NKM_regelbasert_dyp,lys_modellert_copernicus_S3|substrat_grunn_mod...,medium,"MULTIPOLYGON (((970024.167 7942253.331, 970025..."
1,NKM-0000000002,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,afotisk,saltvann,fastbunn,permanent_vanndekket,NKM_regelbasert_dyp,lys_modellert_copernicus_S3|substrat_grunn_mod...,lav,"MULTIPOLYGON (((970100 7942255.093, 970100 794..."
2,NKM-0000000003,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,afotisk,saltvann,fastbunn,permanent_vanndekket,NKM_regelbasert_dyp,lys_modellert_copernicus_S3|substrat_grunn_mod...,medium,"MULTIPOLYGON (((970450 7942263.226, 970450 794..."
3,NKM-0000000004,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,afotisk,saltvann,fastbunn,permanent_vanndekket,NKM_regelbasert_dyp,lys_modellert_copernicus_S3|substrat_grunn_mod...,medium,"MULTIPOLYGON (((967724.85 7942199.9, 967650 79..."
4,NKM-0000000005,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,afotisk,saltvann,fastbunn,permanent_vanndekket,NKM_regelbasert_dyp,lys_modellert_copernicus_S3|substrat_grunn_mod...,medium,"MULTIPOLYGON (((968213.739 7942211.261, 968300..."


In [6]:
fname = mnk.utils.to_filename("nisjemodell", "norge", "2026", CRS.split(":")[1])
out_dir = REPO
gdf_nin_land.to_file(out_dir / f"{fname}.gpkg", driver="GPKG", layer="nin")
gdf_nin_land.to_parquet(out_dir / f"{fname}.geo.parquet", compression="snappy")
print(f"Lagret {len(gdf_nin_land):,} polygoner -> {fname}.gpkg / .geo.parquet")

Lagret 1,099,066 polygoner -> nisjemodell_norge_2026_25833.gpkg / .geo.parquet


In [ ]:
# Upload to PostGIS (optional)
mnk.utils.to_postgis(gdf_nin_land, fname)